# Free-Energy–Driven Quantum-Enhanced Pneumonia Triage from Chest X-Ray Images

This final notebook implements a decision-centric framework for pneumonia triage using compact latent representations, uncertainty quantification, and a hybrid quantum-enhanced transformation. The objective is not only to classify chest X-ray images, but also to determine when predictions are sufficiently reliable for automated decision-making and when they should be deferred for further clinical review.

The notebook is designed for **Google Colab** and follows a reproducible structure. It mounts Google Drive, creates organized output folders, trains a classical baseline and a hybrid quantum model on **PneumoniaMNIST**, evaluates uncertainty and calibration, compares **entropy-based triage** against **free-energy–based triage**, and saves figures, tables, and an aggregated textual summary to Google Drive.

## 1. Environment Setup and Reproducibility

This cell installs required packages, mounts Google Drive when running in Colab, creates output directories, initializes deterministic seeds, and prepares a persistent logging mechanism. All textual outputs are written to `outputs summary.txt`, while figures and tables are saved in dedicated folders.

In [ ]:
# If needed in Colab, uncomment the next line:
# !pip -q install medmnist pennylane scikit-learn

import os
import sys
import json
import time
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.manifold import TSNE

try:
    import pennylane as qml
except Exception as e:
    raise RuntimeError("PennyLane is required. Please install it with: !pip -q install pennylane") from e

try:
    import medmnist
    from medmnist import INFO
except Exception as e:
    raise RuntimeError("medmnist is required. Please install it with: !pip -q install medmnist") from e

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

BASE_OUTPUT = Path("/content/drive/MyDrive/Outputs")
FIG_DIR = BASE_OUTPUT / "figures"
TAB_DIR = BASE_OUTPUT / "tables"
OTH_DIR = BASE_OUTPUT / "others"
for d in [BASE_OUTPUT, FIG_DIR, TAB_DIR, OTH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUTS_SUMMARY = BASE_OUTPUT / "outputs summary.txt"

def log_header(title):
    line = "=" * 100
    msg = f"\n{line}\n{title}\n{line}\n"
    print(msg)
    with open(OUTPUTS_SUMMARY, "a", encoding="utf-8") as f:
        f.write(msg)

def log_text(text=""):
    print(text)
    with open(OUTPUTS_SUMMARY, "a", encoding="utf-8") as f:
        f.write(str(text) + "\n")

def log_json(obj):
    txt = json.dumps(obj, indent=2, default=str)
    print(txt)
    with open(OUTPUTS_SUMMARY, "a", encoding="utf-8") as f:
        f.write(txt + "\n")

with open(OUTPUTS_SUMMARY, "w", encoding="utf-8") as f:
    f.write("STEP 2 FINAL NOTEBOOK LOG\n")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log_header("INITIALIZATION COMPLETE")
log_text(f"IN_COLAB={IN_COLAB}")
log_text(f"BASE_OUTPUT={BASE_OUTPUT}")
log_text(f"FIG_DIR={FIG_DIR}")
log_text(f"TAB_DIR={TAB_DIR}")
log_text(f"OTH_DIR={OTH_DIR}")
log_text(f"OUTPUTS_SUMMARY={OUTPUTS_SUMMARY}")
log_text(f"DEVICE={DEVICE}")

## 2. Configuration

This cell defines the experimental configuration, including dataset choice, model size, training schedule, uncertainty sampling parameters, and free-energy triage parameters. These values can be adjusted for later experiments while preserving the same pipeline.

In [ ]:
CONFIG = {
    "data_flag": "pneumoniamnist",
    "download": True,
    "img_size": 28,
    "batch_size": 64,
    "num_workers": 2,
    "max_train_samples": 3000,
    "max_val_samples": 600,
    "max_test_samples": 600,
    "latent_dim": 8,
    "dropout_p": 0.25,
    "epochs": 6,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "mc_passes": 20,
    "free_energy_lambda": 0.2,
    "triage_quantile": 0.75,
    "quantum_qubits": 4,
    "quantum_layers": 1,
}

log_header("CONFIGURATION")
for k, v in CONFIG.items():
    log_text(f"{k}: {v}")

## 3. Dataset Loading

The notebook uses **PneumoniaMNIST** from the MedMNIST collection. This cell loads train, validation, and test sets, optionally subsamples them for rapid experimentation, and creates PyTorch data loaders.

In [ ]:
from torchvision import transforms

data_flag = CONFIG["data_flag"]
info = INFO[data_flag]
DataClass = getattr(medmnist, info["python_class"])

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = DataClass(split="train", transform=transform, download=CONFIG["download"])
val_dataset   = DataClass(split="val",   transform=transform, download=CONFIG["download"])
test_dataset  = DataClass(split="test",  transform=transform, download=CONFIG["download"])

def maybe_subset(dataset, max_samples):
    if max_samples is None or max_samples >= len(dataset):
        return dataset
    indices = list(range(min(max_samples, len(dataset))))
    return Subset(dataset, indices)

train_dataset = maybe_subset(train_dataset, CONFIG["max_train_samples"])
val_dataset   = maybe_subset(val_dataset,   CONFIG["max_val_samples"])
test_dataset  = maybe_subset(test_dataset,  CONFIG["max_test_samples"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"])
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

log_header("LOADING PNEUMONIAMNIST")
log_text(f"Train samples: {len(train_dataset)}")
log_text(f"Val samples: {len(val_dataset)}")
log_text(f"Test samples: {len(test_dataset)}")

## 4. Visual Inspection of Sample Images

A quick visualization of sample images helps verify that the dataset is correctly loaded and provides a basic sanity check before model training.

In [ ]:
log_header("VISUALIZING SAMPLE IMAGES")

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()

for i in range(10):
    img, label = train_dataset[i]
    axes[i].imshow(img.squeeze().numpy(), cmap="gray")
    axes[i].set_title(f"Label: {int(label)}")
    axes[i].axis("off")

plt.tight_layout()
sample_fig_path = FIG_DIR / "step2_sample_images.png"
plt.savefig(sample_fig_path, dpi=300, bbox_inches="tight")
plt.show()
log_text(f"Saved sample image figure to: {sample_fig_path}")

## 5. Model Definitions

This section defines the encoder, a classical baseline network, and the hybrid quantum model. The hybrid model uses a sample-wise quantum forward pass to avoid batch-shape issues in PennyLane and to ensure stable execution in Colab.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=8, dropout_p=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(32, latent_dim),
        )

    def forward(self, x):
        x = self.features(x)
        z = self.fc(x)
        return z

class BaselineNet(nn.Module):
    def __init__(self, latent_dim=8, dropout_p=0.25):
        super().__init__()
        self.encoder = Encoder(latent_dim=latent_dim, dropout_p=dropout_p)
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(16, 2),
        )

    def forward(self, x, return_latent=False):
        z = self.encoder(x)
        logits = self.classifier(self.dropout(z))
        if return_latent:
            return logits, z
        return logits

class QuantumLayer(nn.Module):
    def __init__(self, n_qubits=4, n_layers=1):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(circuit, weight_shapes)

    def forward(self, x):
        assert x.ndim == 2, f"Quantum input must be [B, Q], got {x.shape}"
        assert x.shape[1] == self.n_qubits, f"Expected {self.n_qubits} qubits, got {x.shape[1]}"
        outputs = [self.qlayer(xi) for xi in x]
        return torch.stack(outputs, dim=0)

class HybridQuantumNet(nn.Module):
    def __init__(self, latent_dim=8, quantum_qubits=4, quantum_layers=1, dropout_p=0.25):
        super().__init__()
        self.encoder = Encoder(latent_dim=latent_dim, dropout_p=dropout_p)
        self.quantum_qubits = quantum_qubits
        self.quantum = QuantumLayer(n_qubits=quantum_qubits, n_layers=quantum_layers)
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim + quantum_qubits, 16),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(16, 2),
        )

    def forward(self, x, return_latent=False):
        z = self.encoder(x)
        z_q = torch.tanh(z[:, :self.quantum_qubits]) * (math.pi / 2.0)
        q = self.quantum(z_q)
        z_tilde = torch.cat([z, q], dim=1)
        logits = self.classifier(self.dropout(z_tilde))
        if return_latent:
            return logits, z_tilde
        return logits

baseline_model = BaselineNet(
    latent_dim=CONFIG["latent_dim"],
    dropout_p=CONFIG["dropout_p"]
).to(DEVICE)

hybrid_model = HybridQuantumNet(
    latent_dim=CONFIG["latent_dim"],
    quantum_qubits=CONFIG["quantum_qubits"],
    quantum_layers=CONFIG["quantum_layers"],
    dropout_p=CONFIG["dropout_p"],
).to(DEVICE)

log_header("DEFINING MODELS")
log_text(str(baseline_model))
log_text(str(hybrid_model))

## 6. Training and Evaluation Utilities

This cell defines the core training, deterministic evaluation, Monte Carlo dropout evaluation, calibration estimation, and triage analysis functions used throughout the notebook.

In [ ]:
def prepare_targets(y):
    if isinstance(y, list):
        y = torch.tensor(y)
    y = y.squeeze()
    if y.ndim == 0:
        y = y.unsqueeze(0)
    return y.long()

def train_one_epoch(model, loader, optimizer):
    model.train()
    losses, preds, probs, targets = [], [], [], []

    for x, y in loader:
        x = x.to(DEVICE)
        y = prepare_targets(y).to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        p = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
        losses.append(loss.item())
        preds.extend(pred.tolist())
        probs.extend(p.tolist())
        targets.extend(y.detach().cpu().numpy().tolist())

    targets = np.array(targets)
    preds = np.array(preds)
    probs = np.array(probs)

    return {
        "loss": float(np.mean(losses)),
        "acc": float(accuracy_score(targets, preds)),
        "f1": float(f1_score(targets, preds, zero_division=0)),
        "auc": float(roc_auc_score(targets, probs)) if len(np.unique(targets)) > 1 else np.nan,
    }

@torch.no_grad()
def evaluate_deterministic(model, loader):
    model.eval()
    losses, preds, probs, targets = [], [], [], []

    for x, y in loader:
        x = x.to(DEVICE)
        y = prepare_targets(y).to(DEVICE)

        logits = model(x)
        loss = F.cross_entropy(logits, y)

        p = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
        losses.append(loss.item())
        preds.extend(pred.tolist())
        probs.extend(p.tolist())
        targets.extend(y.detach().cpu().numpy().tolist())

    targets = np.array(targets)
    preds = np.array(preds)
    probs = np.array(probs)

    return {
        "loss": float(np.mean(losses)),
        "acc": float(accuracy_score(targets, preds)),
        "f1": float(f1_score(targets, preds, zero_division=0)),
        "auc": float(roc_auc_score(targets, probs)) if len(np.unique(targets)) > 1 else np.nan,
        "targets": targets,
        "preds": preds,
        "probs": probs,
    }

def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        idx = bin_ids == b
        if np.any(idx):
            acc = np.mean(y_true[idx] == (y_prob[idx] >= 0.5))
            conf = np.mean(y_prob[idx])
            ece += np.sum(idx) / len(y_true) * abs(acc - conf)
    return float(ece)

def mc_dropout_predict(model, loader, mc_passes=20):
    model.eval()
    enable_dropout(model)

    all_targets = []
    mean_probs_all = []
    var_probs_all = []
    entropy_all = []
    correct_nll_all = []
    latent_all = []
    image_all = []
    pred_all = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = prepare_targets(y).to(DEVICE)

        mc_probs = []
        with torch.no_grad():
            for _ in range(mc_passes):
                logits, latent = model(x, return_latent=True)
                probs = torch.softmax(logits, dim=1)[:, 1]
                mc_probs.append(probs.unsqueeze(0))
            mc_probs = torch.cat(mc_probs, dim=0)

        mean_probs = mc_probs.mean(dim=0)
        var_probs = mc_probs.var(dim=0)
        entropy = -(mean_probs * torch.log(mean_probs + 1e-8) + (1 - mean_probs) * torch.log(1 - mean_probs + 1e-8))
        pred = (mean_probs >= 0.5).long()

        correct_prob = torch.where(y == 1, mean_probs, 1 - mean_probs).clamp_min(1e-8)
        correct_nll = -torch.log(correct_prob)

        all_targets.extend(y.cpu().numpy().tolist())
        mean_probs_all.extend(mean_probs.cpu().numpy().tolist())
        var_probs_all.extend(var_probs.cpu().numpy().tolist())
        entropy_all.extend(entropy.cpu().numpy().tolist())
        correct_nll_all.extend(correct_nll.cpu().numpy().tolist())
        pred_all.extend(pred.cpu().numpy().tolist())
        latent_all.append(latent.detach().cpu().numpy())
        image_all.append(x.detach().cpu().numpy())

    return {
        "targets": np.array(all_targets),
        "mean_probs": np.array(mean_probs_all),
        "var_probs": np.array(var_probs_all),
        "entropy": np.array(entropy_all),
        "correct_nll": np.array(correct_nll_all),
        "preds": np.array(pred_all),
        "latent": np.concatenate(latent_all, axis=0),
        "images": np.concatenate(image_all, axis=0),
    }

def evaluate_uncertainty_and_triage(mc_results, lambda_fe=0.2, triage_quantile=0.75):
    y_true = mc_results["targets"]
    y_prob = mc_results["mean_probs"]
    entropy = mc_results["entropy"]
    variance = mc_results["var_probs"]
    preds = mc_results["preds"]
    correct_nll = mc_results["correct_nll"]

    free_energy = correct_nll + lambda_fe * entropy

    overall_acc = accuracy_score(y_true, preds)
    overall_f1 = f1_score(y_true, preds, zero_division=0)
    overall_auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    ece = expected_calibration_error(y_true, y_prob)

    ent_threshold = np.quantile(entropy, triage_quantile)
    fe_threshold = np.quantile(free_energy, triage_quantile)

    accept_ent = entropy < ent_threshold
    accept_fe = free_energy < fe_threshold

    def summarize_acceptance(mask):
        coverage = float(np.mean(mask))
        if np.sum(mask) == 0:
            return {"coverage": coverage, "risk": np.nan, "accepted_acc": np.nan, "accepted_f1": np.nan, "referred": int(len(mask))}
        accepted_acc = float(accuracy_score(y_true[mask], preds[mask]))
        accepted_f1 = float(f1_score(y_true[mask], preds[mask], zero_division=0))
        risk = float(1.0 - accepted_acc)
        return {
            "coverage": coverage,
            "risk": risk,
            "accepted_acc": accepted_acc,
            "accepted_f1": accepted_f1,
            "referred": int(np.sum(~mask)),
        }

    return {
        "test_acc": float(overall_acc),
        "test_f1": float(overall_f1),
        "test_auc": float(overall_auc),
        "ece": float(ece),
        "mean_entropy": float(np.mean(entropy)),
        "mean_variance": float(np.mean(variance)),
        "mean_free_energy": float(np.mean(free_energy)),
        "entropy_threshold": float(ent_threshold),
        "free_energy_threshold": float(fe_threshold),
        "entropy_triage": summarize_acceptance(accept_ent),
        "free_energy_triage": summarize_acceptance(accept_fe),
        "entropy_values": entropy,
        "free_energy_values": free_energy,
        "accept_entropy": accept_ent,
        "accept_free_energy": accept_fe,
        "probs": y_prob,
        "targets": y_true,
        "preds": preds,
        "latent": mc_results["latent"],
        "images": mc_results["images"],
    }

## 7. Model Training

This section trains the classical baseline and the hybrid quantum model, stores epoch-level histories, and saves these training histories for later reporting.

In [ ]:
log_header("TRAINING MODELS")

models = {
    "baseline": baseline_model,
    "hybrid_quantum": hybrid_model
}

history = {}
history_frames = {}

for model_name, model in models.items():
    log_header(f"TRAINING {model_name.upper()}")
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    history[model_name] = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        start = time.time()
        train_stats = train_one_epoch(model, train_loader, optimizer)
        val_stats = evaluate_deterministic(model, val_loader)
        seconds = time.time() - start

        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "train_acc": train_stats["acc"],
            "train_f1": train_stats["f1"],
            "val_loss": val_stats["loss"],
            "val_acc": val_stats["acc"],
            "val_f1": val_stats["f1"],
            "val_auc": val_stats["auc"],
            "seconds": seconds,
        }
        history[model_name].append(row)
        log_json(row)

    history_frames[model_name] = pd.DataFrame(history[model_name])

log_header("TRAINING HISTORY TABLES")
for model_name, df in history_frames.items():
    log_text(f"\nHistory for {model_name}")
    log_text(df.round(6).to_string(index=False))

## 8. Uncertainty, Calibration, and Triage Evaluation

This cell runs Monte Carlo dropout on the test set, computes uncertainty statistics, calibration error, and compares entropy-based triage with free-energy–based triage.

In [ ]:
log_header("UNCERTAINTY, CALIBRATION, AND TRIAGE EVALUATION")

results = {}

for model_name, model in models.items():
    mc_results = mc_dropout_predict(model, test_loader, mc_passes=CONFIG["mc_passes"])
    summary = evaluate_uncertainty_and_triage(
        mc_results,
        lambda_fe=CONFIG["free_energy_lambda"],
        triage_quantile=CONFIG["triage_quantile"]
    )
    results[model_name] = summary

    compact = {
        "model": model_name,
        "test_acc": summary["test_acc"],
        "test_f1": summary["test_f1"],
        "test_auc": summary["test_auc"],
        "ece": summary["ece"],
        "mean_entropy": summary["mean_entropy"],
        "mean_variance": summary["mean_variance"],
        "mean_free_energy": summary["mean_free_energy"],
        "entropy_threshold": summary["entropy_threshold"],
        "free_energy_threshold": summary["free_energy_threshold"],
        "entropy_triage": summary["entropy_triage"],
        "free_energy_triage": summary["free_energy_triage"],
    }
    log_json(compact)

summary_rows = []
for model_name, summary in results.items():
    summary_rows.append({
        "model": model_name,
        "test_acc": summary["test_acc"],
        "test_f1": summary["test_f1"],
        "test_auc": summary["test_auc"],
        "ece": summary["ece"],
        "mean_entropy": summary["mean_entropy"],
        "mean_variance": summary["mean_variance"],
        "mean_free_energy": summary["mean_free_energy"],
        "entropy_coverage": summary["entropy_triage"]["coverage"],
        "entropy_risk": summary["entropy_triage"]["risk"],
        "entropy_accepted_acc": summary["entropy_triage"]["accepted_acc"],
        "entropy_referred": summary["entropy_triage"]["referred"],
        "free_energy_coverage": summary["free_energy_triage"]["coverage"],
        "free_energy_risk": summary["free_energy_triage"]["risk"],
        "free_energy_accepted_acc": summary["free_energy_triage"]["accepted_acc"],
        "free_energy_referred": summary["free_energy_triage"]["referred"],
    })

summary_df = pd.DataFrame(summary_rows)
log_text("\nFINAL SUMMARY TABLE")
log_text(summary_df.round(6).to_string(index=False))

## 9. Diagnostic Plots

This section generates training curves, calibration plots, and triage plots. All figures are saved to Google Drive for reporting and reuse.

In [ ]:
log_header("PLOTTING FINAL DIAGNOSTIC FIGURES")

plt.figure(figsize=(6, 4))
for model_name, df in history_frames.items():
    plt.plot(df["epoch"], df["val_acc"], marker="o", label=model_name)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Across Epochs")
plt.legend()
plt.tight_layout()
path = FIG_DIR / "step2_val_accuracy.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
log_text(f"Saved: {path}")

def plot_reliability_diagram(y_true, y_prob, title, save_path, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    accuracies = []
    confidences = []
    for i in range(n_bins):
        if i < n_bins - 1:
            idx = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        else:
            idx = (y_prob >= bins[i]) & (y_prob <= bins[i+1])
        if np.any(idx):
            acc = np.mean(y_true[idx] == (y_prob[idx] >= 0.5))
            conf = np.mean(y_prob[idx])
        else:
            acc = np.nan
            conf = np.nan
        accuracies.append(acc)
        confidences.append(conf)

    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.plot(confidences, accuracies, marker="o")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

for model_name, summary in results.items():
    save_path = FIG_DIR / f"step2_reliability_{model_name}.png"
    plot_reliability_diagram(summary["targets"], summary["probs"], f"Reliability Diagram: {model_name}", save_path)
    log_text(f"Saved: {save_path}")

plt.figure(figsize=(6, 4))
for model_name, summary in results.items():
    plt.scatter(summary["entropy_triage"]["coverage"], summary["entropy_triage"]["risk"], marker="o", s=80, label=f"{model_name} | entropy")
    plt.scatter(summary["free_energy_triage"]["coverage"], summary["free_energy_triage"]["risk"], marker="x", s=80, label=f"{model_name} | free-energy")
plt.xlabel("Coverage")
plt.ylabel("Risk")
plt.title("Coverage-Risk Comparison")
plt.legend()
plt.tight_layout()
path = FIG_DIR / "step2_coverage_risk.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
log_text(f"Saved: {path}")

for model_name, summary in results.items():
    plt.figure(figsize=(6, 4))
    plt.scatter(summary["entropy_values"], summary["free_energy_values"], alpha=0.5, s=12)
    plt.xlabel("Entropy")
    plt.ylabel("Free Energy")
    plt.title(f"Entropy vs Free Energy: {model_name}")
    plt.tight_layout()
    save_path = FIG_DIR / f"step2_entropy_vs_free_energy_{model_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    log_text(f"Saved: {save_path}")

log_text("Displayed validation accuracy, calibration, and triage figures.")

## 10. Latent Space Visualization

To better understand the learned geometry, this cell projects latent representations into two dimensions using t-SNE and visualizes class organization for each model.

In [ ]:
log_header("LATENT SPACE VISUALIZATION")

for model_name, summary in results.items():
    latent = summary["latent"]
    targets = summary["targets"]

    max_points = min(400, len(latent))
    latent_sub = latent[:max_points]
    targets_sub = targets[:max_points]

    perplexity = min(30, max(5, max_points // 10))
    latent_2d = TSNE(n_components=2, random_state=SEED, perplexity=perplexity).fit_transform(latent_sub)

    plt.figure(figsize=(6, 5))
    plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=targets_sub, cmap="coolwarm", s=18, alpha=0.8)
    plt.title(f"t-SNE of Latent Space: {model_name}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.tight_layout()
    save_path = FIG_DIR / f"step2_tsne_{model_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    log_text(f"Saved: {save_path}")

## 11. Case-Level Inspection of High-Uncertainty Samples

This final diagnostic step visualizes the most uncertain samples for each model. It helps interpret what kinds of cases are likely to be referred under uncertainty-aware triage.

In [ ]:
log_header("CASE-LEVEL INSPECTION OF MOST UNCERTAIN SAMPLES")

for model_name, summary in results.items():
    entropy = summary["entropy_values"]
    imgs = summary["images"]
    targets = summary["targets"]
    preds = summary["preds"]
    probs = summary["probs"]

    idx = np.argsort(-entropy)[:8]

    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    axes = axes.flatten()

    for ax, i in zip(axes, idx):
        ax.imshow(imgs[i].squeeze(), cmap="gray")
        ax.set_title(f"T={targets[i]} P={preds[i]}\nprob={probs[i]:.2f}\nH={entropy[i]:.3f}")
        ax.axis("off")

    plt.suptitle(f"Most Uncertain Test Samples: {model_name}", y=1.02)
    plt.tight_layout()
    save_path = FIG_DIR / f"step2_uncertain_cases_{model_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    log_text(f"Saved: {save_path}")

log_text("Displayed top uncertain test samples for each model.")

## 12. Persisting Tables and Final Artifacts

This cell saves all main tables to Google Drive, including training histories and final summary tables. It also records a compact narrative summary for external reporting.

In [ ]:
log_header("SAVING FINAL ARTIFACTS")

all_history_df = pd.concat(history_frames.values(), ignore_index=True)
history_csv = TAB_DIR / "step2_history.csv"
all_history_df.to_csv(history_csv, index=False)

summary_csv = TAB_DIR / "step2_summary.csv"
summary_df.to_csv(summary_csv, index=False)

for model_name, summary in results.items():
    detail_df = pd.DataFrame({
        "target": summary["targets"],
        "pred": summary["preds"],
        "prob": summary["probs"],
        "entropy": summary["entropy_values"],
        "free_energy": summary["free_energy_values"],
        "accept_entropy": summary["accept_entropy"],
        "accept_free_energy": summary["accept_free_energy"],
    })
    detail_path = TAB_DIR / f"step2_detail_{model_name}.csv"
    detail_df.to_csv(detail_path, index=False)
    log_text(f"Saved detail table to: {detail_path}")

log_text(f"Saved history CSV to: {history_csv}")
log_text(f"Saved summary CSV to: {summary_csv}")

narrative = {
    "message": "STEP 2 final notebook execution complete.",
    "note": "Free-energy triage and entropy triage were both computed and compared.",
    "best_val_acc_baseline": float(history_frames["baseline"]["val_acc"].max()),
    "best_val_acc_hybrid_quantum": float(history_frames["hybrid_quantum"]["val_acc"].max()),
}
narrative_path = OTH_DIR / "step2_narrative_summary.json"
with open(narrative_path, "w", encoding="utf-8") as f:
    json.dump(narrative, f, indent=2)

log_json(narrative)
log_text(f"Saved narrative summary to: {narrative_path}")